Note: Since GitHub does not execute JavaScript in notebooks, use link in the description to have interative plots

# 1. Data loading and setup

In [1]:
import pandas as pd
import numpy as np
import io
import requests
import lxml
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from scipy.spatial import distance
from IPython.display import HTML, Markdown, display
import warnings

pio.renderers.default = "notebook_connected"
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

### Concatenating test result for all months to one file

The `test_result` table is stored in separate monthly files. First, we filter the data by motorcycle classes (`['test_class_id']` = 1 or 2) and then concatenate the files into a single dataset. Despite the description of `['test_class_id']` in the MOT testing guide, some tests appear in class 1 but not in class 2.

In [2]:
# test_result_list = []
# for i in range(1, 13):
#     month = str(i).zfill(2)
#     file_path = fr"D:\IT\Datasets\MOT\MOT testing data results (2024)\test_result_2024{month}.csv"
#     try:
#         tr = pd.read_csv(
#             file_path, 
#             dtype={'test_id': 'object', 
#                    'vehicle_id': 'object', 
#                    'test_class_id': 'object', 
#                    'test_mileage': 'object', 
#                    'cylinder_capacity': 'object'}, 
#             on_bad_lines='skip'
#         )
#         tr = tr.loc[tr['test_class_id'].str.contains('1|2', regex=True)]
#         test_result_list.append(tr)
#     except FileNotFoundError:
#         print(f"File not found: {file_name}")
# if test_result_list:
#     test_result_full = pd.concat(test_result_list, ignore_index=True)
#     print("Successfully loaded and combined all files.")
# else:
#     print("No files were loaded.");

The concatenated file is saved for convenience.

In [3]:
# # continue with saved motorycle dataset
# test_result_full.to_csv(r'D:\IT\Datasets\MOT\test_result.csv')

### Concatenating 'Test Item' for all months

Next, we do the same for `test_item` table, filter motorcycles from `test_result` table.

In [4]:
# unique_test_id = test_result_full['test_id'].unique()
# test_item_list = []
# for i in range(1, 13):
#     month = str(i).zfill(2)
#     file_path = fr"D:\IT\Datasets\MOT\MOT Testing data failure item (2024)\test_item_2024{month}.csv"
#     try:
#         ti = pd.read_csv(file_path, 
#                          dtype={'test_id': 'object', 
#                                 'rfr_id': 'object', 
#                                 'mot_test_rfr_location_type_id': 'object'}
#                         )
#         ti = ti.loc[ti['test_id'].isin(unique_test_id)]
#         test_item_list.append(ti)
#     except FileNotFoundError:
#         print(f"File not found: {file_path}")
# if test_item_list:
#     test_item_full = pd.concat(test_item_list, ignore_index=True)
#     print("Successfully downloaded and combined all files.")
# else:
#     print("No files were loaded.");

Concatenated `test_item` table is saved to single csv file.

In [5]:
# test_item_full.to_csv(r"D:\IT\Datasets\MOT\test_item.csv")

Concatenated table along with other tables are loaded

In [6]:
test_result_original = pd.read_csv(r"D:\IT\Datasets\MOT\test_result.csv")
test_items_original = pd.read_csv(r"D:\IT\Datasets\MOT\test_item.csv")
item_detail_original = pd.read_csv(r"D:\IT\Datasets\MOT\item_detail.csv", delimiter='|')
item_group_original = pd.read_csv(r"D:\IT\Datasets\MOT\item_group.csv", delimiter='|')

It's better to make copies to ease rollbacks.

In [7]:
tr = test_result_original.copy()
ti = test_items_original.copy()
itde = item_detail_original.copy()
ig = item_group_original.copy()

# 2. Initial Data Exploration and Cleaning

## 'Test Result' table

### Drop columns, duplicates and change dtypes

In [8]:
tr.head()

,Unnamed: 0,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,colour,fuel_type,cylinder_capacity,first_use_date,completed_date
0,0,1731451755,428817244,2024-01-22,1,NT,P,16319.0,W,YAMAHA,XC 125 E VITY,BLUE,PE,124.0,2010-08-21,2024-01-22T10:21:48.000Z
1,1,1165003385,771832290,2024-01-23,1,NT,P,33305.0,NN,HONDA,XL 125 V-4,BLUE,PE,125.0,2004-08-19,2024-01-23T14:03:20.000Z
2,2,635761943,43435464,2024-01-18,2,NT,P,16478.0,B,HONDA,CBR 600 FA-B,WHITE,PE,599.0,2011-12-22,2024-01-18T13:36:28.000Z
3,3,1585139025,1288564542,2024-01-25,2,NT,F,4273.0,SL,PIAGGIO,VESPA,CREAM,PE,200.0,2010-03-10,2024-01-25T11:44:25.000Z
4,4,272083495,1288564542,2024-01-25,2,RT,P,4273.0,SL,PIAGGIO,VESPA,CREAM,PE,200.0,2010-03-10,2024-01-25T17:10:14.000Z


In [9]:
buffer = io.StringIO()
tr.info(buf=buffer)
tr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391115 entries, 0 to 1391114
Data columns (total 16 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Unnamed: 0         1391115 non-null  int64  
 1   test_id            1391115 non-null  int64  
 2   vehicle_id         1391115 non-null  int64  
 3   test_date          1391115 non-null  object 
 4   test_class_id      1391115 non-null  int64  
 5   test_type          1391115 non-null  object 
 6   test_result        1391115 non-null  object 
 7   test_mileage       1362555 non-null  float64
 8   postcode_area      1391115 non-null  object 
 9   make               1391115 non-null  object 
 10  model              1391111 non-null  object 
 11  colour             1391115 non-null  object 
 12  fuel_type          1391115 non-null  object 
 13  cylinder_capacity  1384714 non-null  float64
 14  first_use_date     1391115 non-null  object 
 15  completed_date     1391115 non-n

First, we drop several unnecessary columns. The `['test_class_id']` column is removed because the dataset contains only motorcycles. We remove `['fuel_type']` because 99.87% are type "Petrol" and `['colour']` column.

In [10]:
tr_unnamed_column = tr.iloc[:,0].name
tr = tr.drop(columns=[tr_unnamed_column, 'test_class_id', 'colour', 'fuel_type'])

In [11]:
display(
    Markdown(
        f"Unfortunately, {tr.duplicated().sum()} of {tr.shape[0]} rows are duplicates, "
        f"so they are removed. As we have unique test id for each test we recheck duplicates again."
    )
)
tr = tr.drop_duplicates(keep='last')

Unfortunately, 410547 of 1391115 rows are duplicates, so they are removed. As we have unique test id for each test we recheck duplicates again.

In [12]:
display(
    Markdown(
        f"Still we have {tr.duplicated(subset='test_id').sum()} duplicates. Check what columns have most duplicated values."
    )
)

Still we have 49167 duplicates. Check what columns have most duplicated values.

In [13]:
test_id_dupl = tr[tr.duplicated(subset=['test_id'], keep=False)]
nunique_test_ids = test_id_dupl.groupby('test_id').nunique()
nunique_test_ids.gt(1).sum().sort_values(ascending=False)

first_use_date       46470
model                 1606
test_mileage           758
cylinder_capacity      214
vehicle_id             148
postcode_area          112
make                    68
test_date                0
test_type                0
test_result              0
completed_date           0
dtype: int64

As we don't have duplicates for last 4 columns `(['test_date', 'test_type', 'test_result', 'completed_date'])` we can suppose there were some mistakes with filling columns, the tests are distinct. So we drop duplicates again.

In [14]:
tr = tr.drop_duplicates(subset=['test_id'], keep='last')

Then we assign data types to the columns. ID columns are converted to the "Int32" type.
We remove the timezone from `['completed_date']`, since the dataset only contains data from the UK.
`['test_type']` and `['test_result']` are converted to the "category" type because they contain only 4 and 6 unique values respectively.

In [15]:
date_cols = ['test_date', 'first_use_date', 'completed_date']
for col in date_cols:
    tr[col] = pd.to_datetime(tr[col], errors='coerce')
tr['completed_date'] = tr['completed_date'].dt.tz_localize(None)

int_cols = ['test_mileage', 'cylinder_capacity', 'test_id', 'vehicle_id']
tr[int_cols] = tr[int_cols].astype('Int32')

cat_cols = ['test_type', 'test_result']
tr[cat_cols] = tr[cat_cols].astype('category')

str_cols = tr.select_dtypes(exclude=['Int32', 'datetime', 'category']).columns
tr[str_cols] = tr[str_cols].astype('string')

### Cleaning

#### ['make'] column

In [16]:
brands_lt5 = (tr['make'].value_counts() < 5).sum()
display(
    Markdown(
        f"We have {brands_lt5} of {tr['make'].nunique()} brands with less than 5 motorcycles. After column exploration "
        f"we see that values include not only brand name but model."
    )
)
print(tr.loc[tr['make'].str.contains('HON'), 'make'].value_counts()[10:20])

We have 1451 of 1836 brands with less than 5 motorcycles. After column exploration we see that values include not only brand name but model.

make
HONDA VFR400                            3
HONGYANG HY250-3                        3
HONDA MST260GED                         2
ZHEJIANG HONGZHOU                       2
DIRECT BIKES PYTHON                     2
LIFAN HONGDA LF125                      2
TAIZHOLM ZHONGNENG MOTORCYCLE CO LTD    2
ZHONGYU                                 2
HONDA NX650                             1
HONDA CB400SF-R                         1
Name: count, dtype: Int64


Some cars accidentally appear in the dataset, remove them.

In [17]:
tr = tr.drop(tr.loc[tr['make'].isin(['LOTUS', 'LAND ROVER'])].index)

We have 128 incorrect spelling for Harley-Davidson along with make & model together for some rows. So we edit wrong brand name spelling and convert it to one value and drop model name.

In [18]:
harley = tr.loc[tr['make'].str.contains('HARL')]
print(f"Unique instances of Harley-Davidson spelling: {harley['make'].nunique()}")
print('')
print('Model spelling value count:')
print(harley['make'].value_counts().head(10))

Unique instances of Harley-Davidson spelling: 129

Model spelling value count:
make
HARLEY-DAVIDSON                               28560
HARLEY DAVIDSON                               20748
HARLEY-DAVIDSON IRON 883 XL N 13                 69
HARLEY-DAVIDSON FLSTF FATBOY 1690 12             44
HARLEY-DAVIDSON FAT BOB FXDF 103 1690 13         38
HARLEYDAVIDSON                                   33
HARLEY-DAVIDSON STREET BOB FXDB 1584 13          31
HARLEY-DAVIDSON FORTY EIGHT XL 1200 X 13         29
HARLEY-DAVIDSON HERITAGE STC FLSTC 1690 13       29
HARLEY-DAVIDSON XL 883 N IRON 12                 28
Name: count, dtype: Int64


In [19]:
harley['make_copy'] = harley['make'].str.extract(r'^(HARLE?Y\s?-?DAVID?SON)\s?(.*)$')[0]
harley['make'] = 'HARLEY-DAVIDSON'
harley.drop(columns=['make_copy'], inplace=True)
tr.loc[harley.index] = harley

According to Wikipedia, Indian and Indian Motorcycle refer to one brand, so the names are unified.

In [20]:
print(tr.loc[tr['make'].str.contains('INDIAN'), 'make'].value_counts())
tr.loc[tr['make'].str.contains('INDIAN'), 'make'] = 'INDIAN MOTORCYCLE'

make
INDIAN MOTORCYCLE     2080
INDIAN                 590
INDIAN POLARIS           2
INDIAN MOTORCYCLES       1
INDIAN CHIEF             1
INDIAN SCOUT 60          1
Name: count, dtype: Int64


#### ['model'] column

The state of the column is that we have not only discrete name of the model for particular brand, but names of the series like CBR for `['make']` Honda. As an example, this model series has motorcycles with `['cylinder_capacity']` from 125 to 1000, meaning it includes various types and there is no way to distinct it.

In [21]:
print(tr.loc[(tr['make']=='HONDA') & (tr['model']=='CBR'), 'cylinder_capacity'].value_counts().head(10))
honda_models_num = tr.loc[(tr['make']=='HONDA'), 'model'].nunique()
honda_model_with_one_row = (tr.loc[(tr['make']=='HONDA'), 'model'].value_counts() == 1).sum()

cylinder_capacity
125     2213
649     1803
1000    1325
471      928
599      527
998      470
286      210
999      109
600       94
250       41
Name: count, dtype: Int64


In [22]:
display(
    Markdown(f'At the same time, the column has discrete model names like CBR600F which '
             f'is possibly a part of CBR series. And the most important thing, out '
             f'of {honda_models_num} Honda models, {honda_model_with_one_row} have only one row ' 
             f'so this column is quite hard to analyze therefore to save a memory we drop it '
             f' later but save vehicles IDs for two popular brands with the same cylinder capacity to analyze afterwards.'
    )
)

At the same time, the column has discrete model names like CBR600F which is possibly a part of CBR series. And the most important thing, out of 4029 Honda models, 1933 have only one row so this column is quite hard to analyze therefore to save a memory we drop it  later but save vehicles IDs for two popular brands with the same cylinder capacity to analyze afterwards.

In [23]:
honda_vs_yamaha = tr.loc[tr['model'].isin(['FZS600', 'CBR600F']), 'vehicle_id'].unique()

In [24]:
tr = tr.drop('model', axis=1)

#### ['cylinder_capacity'] column

In [25]:
cc = tr['cylinder_capacity'].dropna()

cc_q1 = cc.quantile(0.25)
cc_median = cc.quantile(0.50)
cc_q3 = cc.quantile(0.75)
cc_iqr = cc_q3 - cc_q1

cc_lower_fence = max(cc.min(), cc_q1 - 1.5 * cc_iqr)
cc_upper_fence = min(cc.max(), cc_q3 + 1.5 * cc_iqr)

cc_outliers = cc[
    (cc < cc_lower_fence) |
    (cc > cc_upper_fence)
]

fig = go.Figure()
fig.add_trace(
    go.Box(
        q1=[cc_q1],
        median=[cc_median],
        q3=[cc_q3],
        lowerfence=[cc_lower_fence],
        upperfence=[cc_upper_fence],
        x=[''],
        boxpoints=False,
        name=''
    )
)

fig.add_trace(
    go.Scatter(
        x=[''] * len(cc_outliers),
        y=cc_outliers,
        mode='markers',
        marker=dict(size=5),
        name='Outliers',
        hovertemplate='Cylinder capacity: %{y}<extra></extra>'
    )
)

fig.update_layout(
    title='Cylinder capacity spread',
    xaxis=dict(
        title='',
        showticklabels=False
    ),
    yaxis=dict(
        title='',
        tickformat=',.0f'
    ),
    width=400,
    height=450,
    showlegend=False
)

fig.show()

The boxplot is used to check the spread of the column, it indicates that `['cylinder_capacity']` (cc) values above 2127 are outliers. Let's dive deeper. A simple fact check reveals that one of the highest cc for motorcycle is 2458 (Triumph Rocket 3) so we flag any value above 2500 and replace "99999" with "np.nan".


In [26]:
tr.loc[tr['cylinder_capacity']==99999, 'cylinder_capacity'] = np.nan

In [27]:
cc_outliers_upperbound = tr.loc[tr['cylinder_capacity']>2500].sort_values(by='cylinder_capacity', ascending=False)
cc_outliers_upperbound.groupby('cylinder_capacity').size().sort_index(ascending=False).head(10)

cylinder_capacity
13585     1
12999     1
12009     1
9976     17
9967      1
9923      1
9488      3
8897      1
8300      1
8200      1
dtype: int64

After a brief investigation, we found a discrepancy in the `[cylinder_capacity]` values. A dot was missing for many values except for Boss Hoss, it has highest cc values in table (5700, 8200), thereafter we will divide other values by 10.

In [28]:
cc_outliers_upperbound = cc_outliers_upperbound.loc[~cc_outliers_upperbound['make'].str.contains('BOSS|V8', regex=True)]

cc_outliers_upperbound['cylinder_capacity'] = (cc_outliers_upperbound['cylinder_capacity'] / 10).round(0)
tr.loc[cc_outliers_upperbound.index, 'cylinder_capacity'] = cc_outliers_upperbound['cylinder_capacity']

Since the lower bound for motorcycles is around 50 cc, values below 40 cc are removed.

In [29]:
tr.loc[tr['cylinder_capacity'] < 40, 'cylinder_capacity'] = np.nan

#### ['test_result'] column

If `['test_result']` is "P"(Passed) or "PRS"(Passed with rectification) it means the vehicle passed the test and won't appear in the table again. Let's check this assumption.

In [30]:
tr = tr.sort_values(by=['vehicle_id', 'completed_date'])
appear_after_pass = pd.DataFrame()
for i in range(4):
    test = tr.groupby('vehicle_id').nth(i)
    if test.empty:
        break
    passed_test = test.loc[lambda x: x['test_result'].isin(['P','PRS']), 'vehicle_id'].values
    passed_but_have_second = (
        tr.loc[tr['vehicle_id'].isin(passed_test)]
        .groupby('vehicle_id', as_index=False)
        .size()
        .sort_values(by='size', ascending=False)
        .loc[lambda x:x['size']>i+1]
    )
    print(f'{passed_but_have_second.shape[0]} vehicles passed {i+1}th test but have next one')
    appear_after_pass = pd.concat([appear_after_pass, passed_but_have_second])

10001 vehicles passed 1th test but have next one
759 vehicles passed 2th test but have next one
24 vehicles passed 3th test but have next one
2 vehicles passed 4th test but have next one


In [31]:
tr = tr.loc[~tr['vehicle_id'].isin(appear_after_pass['vehicle_id'])]

According to test guide, if certain motorcycle part fails the test(has result "F") in categories "Major" or "Dangerous" of `itde['rfr_deficiency_category']` column, then test is considered as failed. Later in analysis, after `tr(Test Result)` table is joined with other tables to obtain test result details, we have few motos, which have "P(Passed)" in `['test_result']` column, though have "F(Failed)" in `itde['rfr_deficiency_category']` column. It's better to delete it.

In [32]:
tr = tr.drop(tr.loc[tr['vehicle_id'].isin([278099932, 323120436, 695051878, 28929412])].index)

#### ['test_type'] column

According to the test guide, first test for each vehicle is labeled as "NT(Full initial test)", if it's second, third and so on, it's "RT(Re-Test)". 

In [33]:
tr['test_type'].value_counts()

test_type
NT    849398
RT     59350
EI         0
ES         0
Name: count, dtype: int64

In [34]:
first_test = tr.sort_values(['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(0).reset_index(drop=True)
first_test['test_type'].value_counts()

test_type
NT    837204
RT       795
EI         0
ES         0
Name: count, dtype: int64

In some cases, the first test is labeled as "RT" (Re-Test). It is unclear whether the previous test failed or took place in 2023, so to avoid discrepancy these tests will be dropped.

In [35]:
tr = tr.loc[~tr['vehicle_id'].isin(first_test.loc[lambda x: x['test_type']=='RT', 'vehicle_id'])]
first_test = tr.sort_values(['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(0).reset_index(drop=True)
first_test['test_type'].value_counts()

test_type
NT    837204
EI         0
ES         0
RT         0
Name: count, dtype: int64

Check other tests

In [36]:
other_test = tr.loc[~tr['test_id'].isin(first_test['test_id'])]
other_test['test_type'].value_counts()

test_type
RT    58553
NT    12189
EI        0
ES        0
Name: count, dtype: int64

We have similar situation for some tests that have type "NT", but their order isn't 1st. They will be dropped as well.

In [37]:
tr = tr.loc[~tr['vehicle_id'].isin(other_test.loc[lambda x: x['test_type']=='NT', 'vehicle_id'])]
first_test = tr.sort_values(['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(0).reset_index(drop=True)
first_test['test_type'].value_counts()

test_type
NT    825346
EI         0
ES         0
RT         0
Name: count, dtype: int64

#### ['first_use_date'] column

According to MOT test guide, vehicles that belong to 1st and 2nd `['test_class_id']` have first test in 3 years after first use, so maximum value for `['first_use_date']` column should be 2021. 

In [38]:
less_than_three_years_vehicles = tr.loc[tr['first_use_date'].dt.year > 2021, 'vehicle_id']
display(Markdown(f'We have {less_than_three_years_vehicles.shape[0]} motos, which were made in 2021 and later, so we drop it.'))
tr = tr.loc[~tr['vehicle_id'].isin(less_than_three_years_vehicles)]

We have 3649 motos, which were made in 2021 and later, so we drop it.

#### Other columns

In [39]:
display(
    Markdown(
        f"As we have only {(tr['test_date'].dt.day != tr['completed_date'].dt.day).sum()} tests "
       f"that weren't completed within a day we have no need in `['test_date']` column."
    )
)
tr = tr.drop('test_date', axis=1)

As we have only 6 tests that weren't completed within a day we have no need in `['test_date']` column.

There are some motorcycles that were produced before 1920, so we check whether the corresponding brands were founded earlier. 

In [40]:
tr.loc[tr.index.isin(tr['first_use_date'].sort_values().index[:10])].sort_values(by='first_use_date')[['make', 'first_use_date']]

,make,first_use_date
393909,LAMBRETTA,1866-08-01
796892,HARLEY-DAVIDSON,1900-01-01
945178,THUMPSTAR,1901-01-01
1081727,SUZUKI,1901-01-01
385893,HARLEY-DAVIDSON,1901-01-01
1295864,QUADRANT,1903-12-31
1233971,MINERVA,1904-12-31
1352725,INDIAN MOTORCYCLE,1914-12-31
901773,IXION,1921-01-14
1047803,RUDGE WHITWORTH,1921-01-24


Harley-Davidson, Lambretta, Suzuki, Thumpstar were founded after 1920, thus we change the dates to "pd.NaT", other brands were founded before indicated date of first use.

In [41]:
tr.loc[
    (tr['first_use_date'] < '1920-01-01') 
    & (tr['make'].isin(['HARLEY-DAVIDSON', 'LAMBRETTA', 'THUMPSTAR', 'SUZUKI']))
] = np.nan

`['Age']` column is created for further analysis.

In [42]:
tr['age'] = ((tr['completed_date'] - tr['first_use_date']).dt.days / 365.25).round(2)

To check `['test_mileage']` column, we add column `['miles_per_year']` to drop outliers.

In [43]:
tr['miles_per_year'] = (tr['test_mileage'] / tr['age']).round(2)
display(Markdown(f'Top 10 test mileage:'))
(tr.loc[
    tr.index.isin(
        tr['test_mileage'].sort_values(ascending=False)
        .index[:10]
        )
    ].sort_values(by='miles_per_year', ascending=False)
    [['first_use_date', 'test_mileage', 'miles_per_year']]
)

Top 10 test mileage:

,first_use_date,test_mileage,miles_per_year
730741,2018-09-07,884216,152978.55
470719,2017-05-20,704364,101056.53
291059,2013-08-01,949647,88918.26
124673,2006-03-14,999999,55679.23
1211538,2012-05-01,657982,52849.96
342587,2005-03-09,999999,52383.39
517024,2001-12-31,929980,41609.84
517026,2001-12-31,929990,41591.68
1176533,1991-02-18,820702,24411.12
1188405,1994-07-23,626639,20790.94


For the upper bound, the value "999999" in `['test_mileage']` seems suspicious and may be a placeholder for missing values. Therefore, it is replaced with "np.nan". Similarly, values higher than 60000 for `['miles_per_year']` are replaced with "np.nan", since it is unrealistic to cover, for instance, 243 miles per day (88938 / 365) for many years on a motorcycle.

In [44]:
tr.loc[(tr['test_mileage'] == 999999) | (tr['miles_per_year'] >= 60000), 'test_mileage'] = np.nan

Simplified version of `['test_result']` column is added for further analysis.

In [45]:
tr['passed'] = tr['test_result'].isin(['P', 'PRS'])
tr = tr.reset_index(drop=True)

In [46]:
tr['test_result'].value_counts()

test_result
P        769122
F         62539
PRS       46738
ABR         371
ABA          41
ABRVE         0
Name: count, dtype: int64

In [47]:
display(Markdown(f"Initial table info:"))
print(buffer.getvalue())

Initial table info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391115 entries, 0 to 1391114
Data columns (total 16 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Unnamed: 0         1391115 non-null  int64  
 1   test_id            1391115 non-null  int64  
 2   vehicle_id         1391115 non-null  int64  
 3   test_date          1391115 non-null  object 
 4   test_class_id      1391115 non-null  int64  
 5   test_type          1391115 non-null  object 
 6   test_result        1391115 non-null  object 
 7   test_mileage       1362555 non-null  float64
 8   postcode_area      1391115 non-null  object 
 9   make               1391115 non-null  object 
 10  model              1391111 non-null  object 
 11  colour             1391115 non-null  object 
 12  fuel_type          1391115 non-null  object 
 13  cylinder_capacity  1384714 non-null  float64
 14  first_use_date     1391115 non-null  object 
 15  completed_date     1391115 non-n

In [48]:
tr = tr.drop(tr.loc[tr['test_id'].isna()].index)
display(Markdown(f"Table info after tranformation:"))
tr.info()

Table info after tranformation:

<class 'pandas.core.frame.DataFrame'>
Index: 878811 entries, 0 to 878815
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   test_id            878811 non-null  Int32         
 1   vehicle_id         878811 non-null  Int32         
 2   test_type          878811 non-null  category      
 3   test_result        878811 non-null  category      
 4   test_mileage       867017 non-null  Int32         
 5   postcode_area      878811 non-null  string        
 6   make               878811 non-null  string        
 7   cylinder_capacity  875186 non-null  Int32         
 8   first_use_date     878809 non-null  datetime64[ns]
 9   completed_date     878811 non-null  datetime64[ns]
 10  age                878809 non-null  float64       
 11  miles_per_year     867022 non-null  Float64       
 12  passed             878811 non-null  bool          
dtypes: Float64(1), Int32(4), bool(1), category(2), da

In [49]:
tr.to_csv(r'D:\IT\Datasets\MOT\cleaned\tr.csv', index=False)

## 'Item Detail' table

In [50]:
itde.head()

,rfr_id,test_class_id,test_item_id,minor_item,rfr_deficiency_category,rfr_desc,rfr_loc_marker,rfr_insp_manual_desc,rfr_advisory_text,test_item_set_section_id
0,4,1,3,Y,Pre-EU Directive,missing,N,an obligatory lamp missing,NaN,1
1,4,2,3,Y,Pre-EU Directive,missing,N,an obligatory lamp missing,NaN,1
2,5,1,3,Y,Pre-EU Directive,damaged and function impaired,N,so damaged or deteriorated that its function i...,damaged but function not impaired,1
3,5,2,3,Y,Pre-EU Directive,damaged and function impaired,N,so damaged or deteriorated that its function i...,damaged but function not impaired,1
4,6,1,3,Y,Pre-EU Directive,deteriorated and function impaired,N,so damaged or deteriorated that its function i...,deteriorated but function not impaired,1


In [51]:
itde.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21069 entries, 0 to 21068
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   rfr_id                    21069 non-null  int64 
 1   test_class_id             21069 non-null  int64 
 2   test_item_id              21069 non-null  int64 
 3   minor_item                21069 non-null  object
 4   rfr_deficiency_category   21069 non-null  object
 5   rfr_desc                  21069 non-null  object
 6   rfr_loc_marker            21069 non-null  object
 7   rfr_insp_manual_desc      20911 non-null  object
 8   rfr_advisory_text         6824 non-null   object
 9   test_item_set_section_id  21069 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 1.6+ MB


In [52]:
buffer = io.StringIO()
itde.info(buf=buffer)

Here we won't filter `['test_class_id']` column, because due to discrepancy we'll have NaN for some rows later if we drop it. 

In [53]:
itde['rfr_insp_manual_desc'] = itde['rfr_insp_manual_desc'].fillna(itde['rfr_desc'])
itde = itde.drop(columns=['test_class_id','rfr_desc','rfr_advisory_text', 'rfr_loc_marker'])
itde.rename(columns={'rfr_insp_manual_desc': 'rfr_desc'}, inplace=True)
itde['minor_item'] = itde['minor_item'].map({'Y': True, 'N': False})

Again, i'll convert all ids column to "Int32" type and columns with low number of unique values to "category".

In [54]:
int_cols = ['rfr_id', 'test_item_id', 'test_item_set_section_id']
itde[int_cols] = itde[int_cols].astype('Int32')
itde['rfr_deficiency_category'] = itde['rfr_deficiency_category'].astype('category')
itde['rfr_desc'] = itde['rfr_desc'].astype('string')

In [55]:
itde = itde.drop_duplicates(keep='first')

In [56]:
display(Markdown(f"Initial table info:"))
print(buffer.getvalue())

Initial table info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21069 entries, 0 to 21068
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   rfr_id                    21069 non-null  int64 
 1   test_class_id             21069 non-null  int64 
 2   test_item_id              21069 non-null  int64 
 3   minor_item                21069 non-null  object
 4   rfr_deficiency_category   21069 non-null  object
 5   rfr_desc                  21069 non-null  object
 6   rfr_loc_marker            21069 non-null  object
 7   rfr_insp_manual_desc      20911 non-null  object
 8   rfr_advisory_text         6824 non-null   object
 9   test_item_set_section_id  21069 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 1.6+ MB



In [57]:
display(Markdown(f"Table info after tranformation:"))
itde.info()

Table info after tranformation:

<class 'pandas.core.frame.DataFrame'>
Index: 7505 entries, 0 to 21067
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   rfr_id                    7505 non-null   Int32   
 1   test_item_id              7505 non-null   Int32   
 2   minor_item                7505 non-null   bool    
 3   rfr_deficiency_category   7505 non-null   category
 4   rfr_desc                  7505 non-null   string  
 5   test_item_set_section_id  7505 non-null   Int32   
dtypes: Int32(3), bool(1), category(1), string(1)
memory usage: 242.1 KB


In [58]:
itde.to_csv(r'D:\IT\Datasets\MOT\cleaned\itde.csv', index=False)

## 'Test Item' table

In [59]:
ti.head()

,Unnamed: 0,test_id,rfr_id,rfr_type_code,mot_test_rfr_location_type_id,dangerous_mark,completed_date
0,0,939637859,40098,A,9,NaN,2024-01-02T10:03:29.000Z
1,1,939637859,40098,A,9,NaN,2024-01-02T10:03:29.000Z
2,2,401677457,40450,A,25,NaN,2024-01-02T10:47:37.000Z
3,3,401677457,40546,A,83,NaN,2024-01-02T10:47:37.000Z
4,4,401677457,40663,A,5,NaN,2024-01-02T10:47:37.000Z


In [60]:
buffer = io.StringIO()
ti.info(buf=buffer)
ti.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815457 entries, 0 to 815456
Data columns (total 7 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   Unnamed: 0                     815457 non-null  int64 
 1   test_id                        815457 non-null  int64 
 2   rfr_id                         815457 non-null  int64 
 3   rfr_type_code                  815457 non-null  object
 4   mot_test_rfr_location_type_id  815457 non-null  int64 
 5   dangerous_mark                 57545 non-null   object
 6   completed_date                 815457 non-null  object
dtypes: int64(4), object(3)
memory usage: 43.6+ MB


Useless columns and duplicates are dropped because each test for vehicle has its own ID.

In [61]:
ti_unnamed_column = ti.iloc[:,0].name
ti.drop(columns=[ti_unnamed_column, 'mot_test_rfr_location_type_id'], inplace=True)
ti = ti.drop_duplicates(keep='first').reset_index(drop=True)

As we have "P(Passed)" value in `tr['test_result']` we change to "P" to "PRS(Pass with rectification)" in `ti['rfr_type_code']`

In [62]:
ti['rfr_type_code'] = ti['rfr_type_code'].str.replace('P', 'PRS')

Data types

In [63]:
ti['completed_date'] = pd.to_datetime(ti['completed_date'])
ti['completed_date']  = ti['completed_date'].dt.tz_localize(None)

int_cols = ['test_id', 'rfr_id']
ti[int_cols] = ti[int_cols].astype('Int32')
ti['rfr_type_code'] = ti['rfr_type_code'].astype('category')

#### ['Dangerous_mark'] column

MOT test data user guide describes this column as a field where a tester could choose to denote that a failure item was dangerous, but it isn't directly comparable to the "Dangerous" category brought in with the EU Roadworthiness Directive from 20th May 2018. So to avoid confusion it is dropped.

In [64]:
ti = ti.drop(columns=['dangerous_mark'])

In [65]:
display(Markdown(f"Initial table info"))
print(buffer.getvalue())

Initial table info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815457 entries, 0 to 815456
Data columns (total 7 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   Unnamed: 0                     815457 non-null  int64 
 1   test_id                        815457 non-null  int64 
 2   rfr_id                         815457 non-null  int64 
 3   rfr_type_code                  815457 non-null  object
 4   mot_test_rfr_location_type_id  815457 non-null  int64 
 5   dangerous_mark                 57545 non-null   object
 6   completed_date                 815457 non-null  object
dtypes: int64(4), object(3)
memory usage: 43.6+ MB



In [66]:
display(Markdown(f"Table info after tranformation"))
ti.info()

Table info after tranformation

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649389 entries, 0 to 649388
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   test_id         649389 non-null  Int32         
 1   rfr_id          649389 non-null  Int32         
 2   rfr_type_code   649389 non-null  category      
 3   completed_date  649389 non-null  datetime64[ns]
dtypes: Int32(2), category(1), datetime64[ns](1)
memory usage: 11.8 MB


In [67]:
ti.to_csv(r'D:\IT\Datasets\MOT\cleaned\ti.csv', index=False)

## 'Item group' table

In [68]:
ig.head()

,test_item_id,test_class_id,parent_id,test_item_set_section_id,item_name
0,0,1,0,0,Vehicle
1,0,2,0,0,Vehicle
2,0,3,0,0,Vehicle
3,0,4,0,0,Vehicle
4,0,5,0,0,Vehicle


In [69]:
ig = ig.loc[ig['test_class_id'].isin([1,2])].drop(columns=['test_class_id'])
int_cols = ['test_item_id', 'parent_id', 'test_item_set_section_id']
ig[int_cols] = ig[int_cols].astype('Int32')

str_cols = ig.select_dtypes(exclude='int').columns
ig[str_cols] = ig[str_cols].astype('string')

In [70]:
ig.info()

<class 'pandas.core.frame.DataFrame'>
Index: 632 entries, 0 to 4135
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   test_item_id              632 non-null    Int32 
 1   parent_id                 632 non-null    Int32 
 2   test_item_set_section_id  632 non-null    Int32 
 3   item_name                 632 non-null    string
dtypes: Int32(3), string(1)
memory usage: 19.1 KB


In [71]:
ig.to_csv(r'D:\IT\Datasets\MOT\cleaned\ig.csv', index=False)